<a href="https://colab.research.google.com/github/KingLeolll/Estudo-de-Descargas-Parciais-em-Barras-Geradoras-com-Pyhton-e-Elmer/blob/main/Compara%C3%A7%C3%A3o_de_Defeitos_com_Quantifica%C3%A7%C3%A3o_em_Unidades_Relativas_(RU).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Comparação de Defeitos com Quantificação em Unidades Relativas (RU)
Conforme Figura 2 e Tabela I do Artigo

Compara 3 tipos de defeitos com medições em RU:
- C-1A: Cavidade Interna (0 RU - sem DP)
- CE-1E: Descarga Superficial (1.1 RU)
- CE-1D: Corona (5.6 RU)
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import warnings

warnings.filterwarnings('ignore')

# Importar simulador
import sys
sys.path.insert(0, '/home/ubuntu')

from pd_simulation_final import PDSimulatorWithRUQuantification, PRPDMeasurement


class DefectComparisonWithRU:
    """Análise comparativa de defeitos com quantificação em RU"""

    # Tabela I do artigo: Correntes medidas em RU
    TABLE_I_REFERENCE = {
        'internal_void': {
            'label': 'C-1A',
            'description': 'Cavidade Interna',
            'reference_ru': 0.0,
        },
        'surface_discharge': {
            'label': 'CE-1E',
            'description': 'Descarga Superficial',
            'reference_ru': 1.1,
        },
        'corona_point': {
            'label': 'CE-1D',
            'description': 'Corona (Ponta-Plano)',
            'reference_ru': 5.6,
        }
    }

    def __init__(self):
        self.defect_types = ['internal_void', 'surface_discharge', 'corona_point']
        self.simulators = {
            dtype: PDSimulatorWithRUQuantification(dtype)
            for dtype in self.defect_types
        }
        self.results = {}

    def run_comparison(self, E_peak: float, frequency: float, num_cycles: int) -> Dict:
        """Executa simulação comparativa com quantificação em RU"""

        print(f"\n{'='*80}")
        print(f"COMPARAÇÃO DE DEFEITOS - QUANTIFICAÇÃO EM UNIDADES RELATIVAS (RU)")
        print(f"Conforme Tabela I do Artigo")
        print(f"{'='*80}\n")

        comparison_results = {}

        for defect_type in self.defect_types:
            print(f"\n{'─'*80}")
            print(f"Simulando: {defect_type.upper()}")
            print(f"{'─'*80}")

            simulator = self.simulators[defect_type]
            measurements, avg_current_ru = simulator.simulate(E_peak, frequency, num_cycles)

            reference_info = self.TABLE_I_REFERENCE[defect_type]
            reference_ru = reference_info['reference_ru']

            # Calcular estatísticas
            if measurements:
                currents = [m.current_ru for m in measurements]
                phases = [m.phase for m in measurements]

                positive_events = [m for m in measurements if 0 <= m.phase <= 180]
                negative_events = [m for m in measurements if 180 < m.phase <= 360]

                stats = {
                    'total_events': len(measurements),
                    'avg_current_ru': avg_current_ru,
                    'max_current_ru': max(currents),
                    'min_current_ru': min(currents),
                    'std_current_ru': np.std(currents),
                    'positive_count': len(positive_events),
                    'negative_count': len(negative_events),
                    'asymmetry_ratio': len(negative_events) / len(positive_events) if positive_events else 0,
                    'positive_avg_ru': np.mean([m.current_ru for m in positive_events]) if positive_events else 0,
                    'negative_avg_ru': np.mean([m.current_ru for m in negative_events]) if negative_events else 0,
                }
            else:
                stats = {
                    'total_events': 0,
                    'avg_current_ru': 0,
                    'max_current_ru': 0,
                    'min_current_ru': 0,
                    'std_current_ru': 0,
                    'positive_count': 0,
                    'negative_count': 0,
                    'asymmetry_ratio': 0,
                    'positive_avg_ru': 0,
                    'negative_avg_ru': 0,
                }

            comparison_results[defect_type] = {
                'measurements': measurements,
                'avg_current_ru': avg_current_ru,
                'reference_ru': reference_ru,
                'statistics': stats,
                'reference_info': reference_info,
            }

            # Imprimir resultados
            print(f"\nTabela I - Slice: {reference_info['label']}")
            print(f"Descrição: {reference_info['description']}")
            print(f"\nQuantificação de Corrente em RU:")
            print(f"  Corrente Medida: {avg_current_ru:.2f} RU")
            print(f"  Referência (Artigo): {reference_ru:.2f} RU")
            print(f"  Diferença: {abs(avg_current_ru - reference_ru):.2f} RU")
            print(f"  Erro Relativo: {abs(avg_current_ru - reference_ru)/max(reference_ru, 0.1)*100:.1f}%")

            print(f"\nEstatísticas de DP:")
            print(f"  Total de Eventos: {stats['total_events']}")
            print(f"  Corrente Máxima: {stats['max_current_ru']:.2f} RU")
            print(f"  Corrente Mínima: {stats['min_current_ru']:.2f} RU")
            print(f"  Desvio Padrão: {stats['std_current_ru']:.2f} RU")
            print(f"  Eventos Positivos: {stats['positive_count']} (Média: {stats['positive_avg_ru']:.2f} RU)")
            print(f"  Eventos Negativos: {stats['negative_count']} (Média: {stats['negative_avg_ru']:.2f} RU)")
            print(f"  Razão de Assimetria (Neg/Pos): {stats['asymmetry_ratio']:.2f}")

        self.results = comparison_results
        return comparison_results

    def plot_comparison(self, E_peak: float, frequency: float, num_cycles: int):
        """Plota comparação visual entre defeitos (similar à Figura 2)"""

        fig = plt.figure(figsize=(18, 12))

        # Plotar cada defeito
        for idx, defect_type in enumerate(self.defect_types):
            result = self.results[defect_type]
            measurements = result['measurements']
            avg_current_ru = result['avg_current_ru']
            reference_ru = result['reference_ru']
            reference_info = result['reference_info']

            # Gráfico 1: PRPD (similar à Figura 2 esquerda/direita)
            ax = plt.subplot(3, 4, idx*4 + 1)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                # Separar semiciclos
                pos_phases = [m.phase for m in measurements if 0 <= m.phase <= 180]
                pos_currents = [m.current_ru for m in measurements if 0 <= m.phase <= 180]
                neg_phases = [m.phase for m in measurements if 180 < m.phase <= 360]
                neg_currents = [m.current_ru for m in measurements if 180 < m.phase <= 360]

                ax.scatter(pos_phases, pos_currents, color='red', alpha=0.6, s=25, label='Pos')
                ax.scatter(neg_phases, neg_currents, color='blue', alpha=0.6, s=25, label='Neg')

            # Adicionar tensão AC
            phase_range = np.linspace(0, 360, 1000)
            voltage = np.sin(np.radians(phase_range))
            ax_twin = ax.twinx()
            ax_twin.plot(phase_range, voltage, 'k--', linewidth=1, alpha=0.3)

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente (RU)', fontsize=10, color='black')
            ax_twin.set_ylabel('Tensão (p.u.)', fontsize=9, color='gray')
            ax.set_title(f'{reference_info["label"]}: {reference_info["description"]}\nPRPD (Figura 2)',
                        fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 360)
            ax.legend(loc='upper right', fontsize=8)

            # Gráfico 2: Histograma de Fase com RU
            ax = plt.subplot(3, 4, idx*4 + 2)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                phase_bins = np.linspace(0, 360, 37)
                hist_ru, _ = np.histogram(phases, bins=phase_bins, weights=currents)

                ax.bar(phase_bins[:-1], hist_ru, width=10, color='green', alpha=0.7, edgecolor='black')

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente Integrada (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nHistograma de Corrente', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')

            # Gráfico 3: Tabela I - Comparação de RU
            ax = plt.subplot(3, 4, idx*4 + 3)

            categories = ['Medido', 'Referência\n(Tabela I)']
            values = [avg_current_ru, reference_ru]
            colors = ['#3498db', '#e74c3c']

            bars = ax.bar(categories, values, color=colors, alpha=0.8, edgecolor='black', width=0.6)

            # Adicionar valores nas barras
            for bar, value in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.2f}\nRU',
                       ha='center', va='bottom', fontsize=11, fontweight='bold')

            ax.set_ylabel('Corrente (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nTabela I do Artigo', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(0, max(values) * 1.3)

            # Gráfico 4: Estatísticas detalhadas
            ax = plt.subplot(3, 4, idx*4 + 4)
            ax.axis('off')

            stats = result['statistics']
            stats_text = f"""
{reference_info['label']} - {reference_info['description']}

TABELA I - QUANTIFICAÇÃO EM RU:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I (RU) Medido: {avg_current_ru:.2f}
I (RU) Referência: {reference_ru:.2f}
Diferença: {abs(avg_current_ru - reference_ru):.2f}

ESTATÍSTICAS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total de Eventos: {stats['total_events']}
Corrente Máxima: {stats['max_current_ru']:.2f} RU
Corrente Mínima: {stats['min_current_ru']:.2f} RU
Desvio Padrão: {stats['std_current_ru']:.2f} RU

SEMICICLOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Positivos: {stats['positive_count']}
  Média: {stats['positive_avg_ru']:.2f} RU
Negativos: {stats['negative_count']}
  Média: {stats['negative_avg_ru']:.2f} RU
Assimetria: {stats['asymmetry_ratio']:.2f}
            """

            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                   fontsize=8.5, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.suptitle(
            f'Comparação de Defeitos - Quantificação em Unidades Relativas (RU)\n'
            f'Conforme Figura 2 e Tabela I do Artigo\n'
            f'E_peak: {E_peak/1e6:.1f} MV/m | Freq: {frequency} Hz | Ciclos: {num_cycles}',
            fontsize=13, fontweight='bold', y=0.995
        )

        plt.tight_layout(rect=[0, 0, 1, 0.99])
        plt.savefig('defect_comparison_ru_quantification.png', dpi=150, bbox_inches='tight')
        print("\n✓ Gráfico salvo: defect_comparison_ru_quantification.png")
        plt.show()

    def generate_table_i_report(self):
        """Gera relatório formatado como Tabela I do artigo"""

        print(f"\n{'='*80}")
        print(f"TABELA I - CORRENTE EM UNIDADES RELATIVAS (RU)")
        print(f"Medições de Descarga Parcial com Antena EM")
        print(f"{'='*80}\n")

        # Cabeçalho da tabela
        print(f"{'Slice':<20} {'C-1A':<15} {'CE-1E':<15} {'CE-1D':<15}")
        print(f"{'Descrição':<20} {'Cavidade Int.':<15} {'Desc. Superf.':<15} {'Corona':<15}")
        print("─" * 80)

        # Linha de corrente medida
        print(f"{'I (RU) - Medido':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['avg_current_ru']:<15.2f}", end='')
        print()

        # Linha de referência
        print(f"{'I (RU) - Referência':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['reference_ru']:<15.2f}", end='')
        print()

        # Linha de diferença
        print(f"{'Diferença':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            diff = abs(result['avg_current_ru'] - result['reference_ru'])
            print(f"{diff:<15.2f}", end='')
        print()

        print("\n" + "─" * 80)
        print("\nINTERPRETAÇÃO (Conforme Seção III do Artigo):\n")

        # Encontrar maior corrente
        max_current = max(
            (self.results[dt]['avg_current_ru'], dt)
            for dt in self.defect_types
        )

        print(f"• Maior Atividade de DP: {max_current[1]} ({max_current[0]:.2f} RU)")
        print(f"  → Maior concentração de corrente de descarga")

        # Encontrar maior assimetria
        max_asym = max(
            (self.results[dt]['statistics']['asymmetry_ratio'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Assimetria: {max_asym[1]} ({max_asym[0]:.2f})")
        print(f"  → Maior concentração de eventos em semiciclo negativo")

        # Encontrar maior número de eventos
        max_events = max(
            (self.results[dt]['statistics']['total_events'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Número de Eventos: {max_events[1]} ({max_events[0]} eventos)")
        print(f"  → Maior frequência de descargas parciais")


if __name__ == '__main__':
    print("╔════════════════════════════════════════════════════════════╗")
    print("║  Comparação de Defeitos - Quantificação em RU             ║")
    print("║  Conforme Tabela I do Artigo                              ║")
    print("╚════════════════════════════════════════════════════════════╝")

    # Parâmetros
    E_peak = 4.0e6  # V/m
    frequency = 60  # Hz
    num_cycles = 100

    # Executar análise
    analyzer = DefectComparisonWithRU()
    results = analyzer.run_comparison(E_peak, frequency, num_cycles)

    # Plotar comparação
    analyzer.plot_comparison(E_peak, frequency, num_cycles)

    # Gerar Tabela I
    analyzer.generate_table_i_report()

    print(f"\n{'='*80}")
    print("✓ Análise concluída com sucesso!")
    print(f"{'='*80}\n")

In [ ]:
"""
Comparação de Defeitos com Quantificação em Unidades Relativas (RU)
Conforme Figura 2 e Tabela I do Artigo

Compara 3 tipos de defeitos com medições em RU:
- C-1A: Cavidade Interna (0 RU - sem DP)
- CE-1E: Descarga Superficial (1.1 RU)
- CE-1D: Corona (5.6 RU)
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import warnings

warnings.filterwarnings('ignore')

# Importar simulador
import sys
sys.path.insert(0, '/home/ubuntu')

from pd_simulation_final import PDSimulatorWithRUQuantification, PRPDMeasurement


class DefectComparisonWithRU:
    """Análise comparativa de defeitos com quantificação em RU"""

    # Tabela I do artigo: Correntes medidas em RU
    TABLE_I_REFERENCE = {
        'internal_void': {
            'label': 'C-1A',
            'description': 'Cavidade Interna',
            'reference_ru': 0.0,
        },
        'surface_discharge': {
            'label': 'CE-1E',
            'description': 'Descarga Superficial',
            'reference_ru': 1.1,
        },
        'corona_point': {
            'label': 'CE-1D',
            'description': 'Corona (Ponta-Plano)',
            'reference_ru': 5.6,
        }
    }

    def __init__(self):
        self.defect_types = ['internal_void', 'surface_discharge', 'corona_point']
        self.simulators = {
            dtype: PDSimulatorWithRUQuantification(dtype)
            for dtype in self.defect_types
        }
        self.results = {}

    def run_comparison(self, E_peak: float, frequency: float, num_cycles: int) -> Dict:
        """Executa simulação comparativa com quantificação em RU"""

        print(f"\n{'='*80}")
        print(f"COMPARAÇÃO DE DEFEITOS - QUANTIFICAÇÃO EM UNIDADES RELATIVAS (RU)")
        print(f"Conforme Tabela I do Artigo")
        print(f"{'='*80}\n")

        comparison_results = {}

        for defect_type in self.defect_types:
            print(f"\n{'─'*80}")
            print(f"Simulando: {defect_type.upper()}")
            print(f"{'─'*80}")

            simulator = self.simulators[defect_type]
            measurements, avg_current_ru = simulator.simulate(E_peak, frequency, num_cycles)

            reference_info = self.TABLE_I_REFERENCE[defect_type]
            reference_ru = reference_info['reference_ru']

            # Calcular estatísticas
            if measurements:
                currents = [m.current_ru for m in measurements]
                phases = [m.phase for m in measurements]

                positive_events = [m for m in measurements if 0 <= m.phase <= 180]
                negative_events = [m for m in measurements if 180 < m.phase <= 360]

                stats = {
                    'total_events': len(measurements),
                    'avg_current_ru': avg_current_ru,
                    'max_current_ru': max(currents),
                    'min_current_ru': min(currents),
                    'std_current_ru': np.std(currents),
                    'positive_count': len(positive_events),
                    'negative_count': len(negative_events),
                    'asymmetry_ratio': len(negative_events) / len(positive_events) if positive_events else 0,
                    'positive_avg_ru': np.mean([m.current_ru for m in positive_events]) if positive_events else 0,
                    'negative_avg_ru': np.mean([m.current_ru for m in negative_events]) if negative_events else 0,
                }
            else:
                stats = {
                    'total_events': 0,
                    'avg_current_ru': 0,
                    'max_current_ru': 0,
                    'min_current_ru': 0,
                    'std_current_ru': 0,
                    'positive_count': 0,
                    'negative_count': 0,
                    'asymmetry_ratio': 0,
                    'positive_avg_ru': 0,
                    'negative_avg_ru': 0,
                }

            comparison_results[defect_type] = {
                'measurements': measurements,
                'avg_current_ru': avg_current_ru,
                'reference_ru': reference_ru,
                'statistics': stats,
                'reference_info': reference_info,
            }

            # Imprimir resultados
            print(f"\nTabela I - Slice: {reference_info['label']}")
            print(f"Descrição: {reference_info['description']}")
            print(f"\nQuantificação de Corrente em RU:")
            print(f"  Corrente Medida: {avg_current_ru:.2f} RU")
            print(f"  Referência (Artigo): {reference_ru:.2f} RU")
            print(f"  Diferença: {abs(avg_current_ru - reference_ru):.2f} RU")
            print(f"  Erro Relativo: {abs(avg_current_ru - reference_ru)/max(reference_ru, 0.1)*100:.1f}%")

            print(f"\nEstatísticas de DP:")
            print(f"  Total de Eventos: {stats['total_events']}")
            print(f"  Corrente Máxima: {stats['max_current_ru']:.2f} RU")
            print(f"  Corrente Mínima: {stats['min_current_ru']:.2f} RU")
            print(f"  Desvio Padrão: {stats['std_current_ru']:.2f} RU")
            print(f"  Eventos Positivos: {stats['positive_count']} (Média: {stats['positive_avg_ru']:.2f} RU)")
            print(f"  Eventos Negativos: {stats['negative_count']} (Média: {stats['negative_avg_ru']:.2f} RU)")
            print(f"  Razão de Assimetria (Neg/Pos): {stats['asymmetry_ratio']:.2f}")

        self.results = comparison_results
        return comparison_results

    def plot_comparison(self, E_peak: float, frequency: float, num_cycles: int):
        """Plota comparação visual entre defeitos (similar à Figura 2)"""

        fig = plt.figure(figsize=(18, 12))

        # Plotar cada defeito
        for idx, defect_type in enumerate(self.defect_types):
            result = self.results[defect_type]
            measurements = result['measurements']
            avg_current_ru = result['avg_current_ru']
            reference_ru = result['reference_ru']
            reference_info = result['reference_info']

            # Gráfico 1: PRPD (similar à Figura 2 esquerda/direita)
            ax = plt.subplot(3, 4, idx*4 + 1)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                # Separar semiciclos
                pos_phases = [m.phase for m in measurements if 0 <= m.phase <= 180]
                pos_currents = [m.current_ru for m in measurements if 0 <= m.phase <= 180]
                neg_phases = [m.phase for m in measurements if 180 < m.phase <= 360]
                neg_currents = [m.current_ru for m in measurements if 180 < m.phase <= 360]

                ax.scatter(pos_phases, pos_currents, color='red', alpha=0.6, s=25, label='Pos')
                ax.scatter(neg_phases, neg_currents, color='blue', alpha=0.6, s=25, label='Neg')

            # Adicionar tensão AC
            phase_range = np.linspace(0, 360, 1000)
            voltage = np.sin(np.radians(phase_range))
            ax_twin = ax.twinx()
            ax_twin.plot(phase_range, voltage, 'k--', linewidth=1, alpha=0.3)

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente (RU)', fontsize=10, color='black')
            ax_twin.set_ylabel('Tensão (p.u.)', fontsize=9, color='gray')
            ax.set_title(f'{reference_info["label"]}: {reference_info["description"]}\nPRPD (Figura 2)',
                        fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 360)
            ax.legend(loc='upper right', fontsize=8)

            # Gráfico 2: Histograma de Fase com RU
            ax = plt.subplot(3, 4, idx*4 + 2)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                phase_bins = np.linspace(0, 360, 37)
                hist_ru, _ = np.histogram(phases, bins=phase_bins, weights=currents)

                ax.bar(phase_bins[:-1], hist_ru, width=10, color='green', alpha=0.7, edgecolor='black')

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente Integrada (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nHistograma de Corrente', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')

            # Gráfico 3: Tabela I - Comparação de RU
            ax = plt.subplot(3, 4, idx*4 + 3)

            categories = ['Medido', 'Referência\n(Tabela I)']
            values = [avg_current_ru, reference_ru]
            colors = ['#3498db', '#e74c3c']

            bars = ax.bar(categories, values, color=colors, alpha=0.8, edgecolor='black', width=0.6)

            # Adicionar valores nas barras
            for bar, value in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.2f}\nRU',
                       ha='center', va='bottom', fontsize=11, fontweight='bold')

            ax.set_ylabel('Corrente (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nTabela I do Artigo', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(0, max(values) * 1.3)

            # Gráfico 4: Estatísticas detalhadas
            ax = plt.subplot(3, 4, idx*4 + 4)
            ax.axis('off')

            stats = result['statistics']
            stats_text = f"""
{reference_info['label']} - {reference_info['description']}

TABELA I - QUANTIFICAÇÃO EM RU:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I (RU) Medido: {avg_current_ru:.2f}
I (RU) Referência: {reference_ru:.2f}
Diferença: {abs(avg_current_ru - reference_ru):.2f}

ESTATÍSTICAS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total de Eventos: {stats['total_events']}
Corrente Máxima: {stats['max_current_ru']:.2f} RU
Corrente Mínima: {stats['min_current_ru']:.2f} RU
Desvio Padrão: {stats['std_current_ru']:.2f} RU

SEMICICLOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Positivos: {stats['positive_count']}
  Média: {stats['positive_avg_ru']:.2f} RU
Negativos: {stats['negative_count']}
  Média: {stats['negative_avg_ru']:.2f} RU
Assimetria: {stats['asymmetry_ratio']:.2f}
            """

            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                   fontsize=8.5, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.suptitle(
            f'Comparação de Defeitos - Quantificação em Unidades Relativas (RU)\n'
            f'Conforme Figura 2 e Tabela I do Artigo\n'
            f'E_peak: {E_peak/1e6:.1f} MV/m | Freq: {frequency} Hz | Ciclos: {num_cycles}',
            fontsize=13, fontweight='bold', y=0.995
        )

        plt.tight_layout(rect=[0, 0, 1, 0.99])
        plt.savefig('defect_comparison_ru_quantification.png', dpi=150, bbox_inches='tight')
        print("\n✓ Gráfico salvo: defect_comparison_ru_quantification.png")
        plt.show()

    def generate_table_i_report(self):
        """Gera relatório formatado como Tabela I do artigo"""

        print(f"\n{'='*80}")
        print(f"TABELA I - CORRENTE EM UNIDADES RELATIVAS (RU)")
        print(f"Medições de Descarga Parcial com Antena EM")
        print(f"{'='*80}\n")

        # Cabeçalho da tabela
        print(f"{'Slice':<20} {'C-1A':<15} {'CE-1E':<15} {'CE-1D':<15}")
        print(f"{'Descrição':<20} {'Cavidade Int.':<15} {'Desc. Superf.':<15} {'Corona':<15}")
        print("─" * 80)

        # Linha de corrente medida
        print(f"{'I (RU) - Medido':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['avg_current_ru']:<15.2f}", end='')
        print()

        # Linha de referência
        print(f"{'I (RU) - Referência':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['reference_ru']:<15.2f}", end='')
        print()

        # Linha de diferença
        print(f"{'Diferença':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            diff = abs(result['avg_current_ru'] - result['reference_ru'])
            print(f"{diff:<15.2f}", end='')
        print()

        print("\n" + "─" * 80)
        print("\nINTERPRETAÇÃO (Conforme Seção III do Artigo):\n")

        # Encontrar maior corrente
        max_current = max(
            (self.results[dt]['avg_current_ru'], dt)
            for dt in self.defect_types
        )

        print(f"• Maior Atividade de DP: {max_current[1]} ({max_current[0]:.2f} RU)")
        print(f"  → Maior concentração de corrente de descarga")

        # Encontrar maior assimetria
        max_asym = max(
            (self.results[dt]['statistics']['asymmetry_ratio'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Assimetria: {max_asym[1]} ({max_asym[0]:.2f})")
        print(f"  → Maior concentração de eventos em semiciclo negativo")

        # Encontrar maior número de eventos
        max_events = max(
            (self.results[dt]['statistics']['total_events'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Número de Eventos: {max_events[1]} ({max_events[0]} eventos)")
        print(f"  → Maior frequência de descargas parciais")


if __name__ == '__main__':
    print("╔════════════════════════════════════════════════════════════╗")
    print("║  Comparação de Defeitos - Quantificação em RU             ║")
    print("║  Conforme Tabela I do Artigo                              ║")
    print("╚════════════════════════════════════════════════════════════╝")

    # Parâmetros
    E_peak = 4.0e6  # V/m
    frequency = 60  # Hz
    num_cycles = 100

    # Executar análise
    analyzer = DefectComparisonWithRU()
    results = analyzer.run_comparison(E_peak, frequency, num_cycles)

    # Plotar comparação
    analyzer.plot_comparison(E_peak, frequency, num_cycles)

    # Gerar Tabela I
    analyzer.generate_table_i_report()

    print(f"\n{'='*80}")
    print("✓ Análise concluída com sucesso!")
    print(f"{'='*80}\n")

In [ ]:
"""
Comparação de Defeitos com Quantificação em Unidades Relativas (RU)
Conforme Figura 2 e Tabela I do Artigo

Compara 3 tipos de defeitos com medições em RU:
- C-1A: Cavidade Interna (0 RU - sem DP)
- CE-1E: Descarga Superficial (1.1 RU)
- CE-1D: Corona (5.6 RU)
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import warnings

warnings.filterwarnings('ignore')

# Importar simulador
import sys
sys.path.insert(0, '/home/ubuntu')

from pd_simulation_final import PDSimulatorWithRUQuantification, PRPDMeasurement


class DefectComparisonWithRU:
    """Análise comparativa de defeitos com quantificação em RU"""

    # Tabela I do artigo: Correntes medidas em RU
    TABLE_I_REFERENCE = {
        'internal_void': {
            'label': 'C-1A',
            'description': 'Cavidade Interna',
            'reference_ru': 0.0,
        },
        'surface_discharge': {
            'label': 'CE-1E',
            'description': 'Descarga Superficial',
            'reference_ru': 1.1,
        },
        'corona_point': {
            'label': 'CE-1D',
            'description': 'Corona (Ponta-Plano)',
            'reference_ru': 5.6,
        }
    }

    def __init__(self):
        self.defect_types = ['internal_void', 'surface_discharge', 'corona_point']
        self.simulators = {
            dtype: PDSimulatorWithRUQuantification(dtype)
            for dtype in self.defect_types
        }
        self.results = {}

    def run_comparison(self, E_peak: float, frequency: float, num_cycles: int) -> Dict:
        """Executa simulação comparativa com quantificação em RU"""

        print(f"\n{'='*80}")
        print(f"COMPARAÇÃO DE DEFEITOS - QUANTIFICAÇÃO EM UNIDADES RELATIVAS (RU)")
        print(f"Conforme Tabela I do Artigo")
        print(f"{'='*80}\n")

        comparison_results = {}

        for defect_type in self.defect_types:
            print(f"\n{'─'*80}")
            print(f"Simulando: {defect_type.upper()}")
            print(f"{'─'*80}")

            simulator = self.simulators[defect_type]
            measurements, avg_current_ru = simulator.simulate(E_peak, frequency, num_cycles)

            reference_info = self.TABLE_I_REFERENCE[defect_type]
            reference_ru = reference_info['reference_ru']

            # Calcular estatísticas
            if measurements:
                currents = [m.current_ru for m in measurements]
                phases = [m.phase for m in measurements]

                positive_events = [m for m in measurements if 0 <= m.phase <= 180]
                negative_events = [m for m in measurements if 180 < m.phase <= 360]

                stats = {
                    'total_events': len(measurements),
                    'avg_current_ru': avg_current_ru,
                    'max_current_ru': max(currents),
                    'min_current_ru': min(currents),
                    'std_current_ru': np.std(currents),
                    'positive_count': len(positive_events),
                    'negative_count': len(negative_events),
                    'asymmetry_ratio': len(negative_events) / len(positive_events) if positive_events else 0,
                    'positive_avg_ru': np.mean([m.current_ru for m in positive_events]) if positive_events else 0,
                    'negative_avg_ru': np.mean([m.current_ru for m in negative_events]) if negative_events else 0,
                }
            else:
                stats = {
                    'total_events': 0,
                    'avg_current_ru': 0,
                    'max_current_ru': 0,
                    'min_current_ru': 0,
                    'std_current_ru': 0,
                    'positive_count': 0,
                    'negative_count': 0,
                    'asymmetry_ratio': 0,
                    'positive_avg_ru': 0,
                    'negative_avg_ru': 0,
                }

            comparison_results[defect_type] = {
                'measurements': measurements,
                'avg_current_ru': avg_current_ru,
                'reference_ru': reference_ru,
                'statistics': stats,
                'reference_info': reference_info,
            }

            # Imprimir resultados
            print(f"\nTabela I - Slice: {reference_info['label']}")
            print(f"Descrição: {reference_info['description']}")
            print(f"\nQuantificação de Corrente em RU:")
            print(f"  Corrente Medida: {avg_current_ru:.2f} RU")
            print(f"  Referência (Artigo): {reference_ru:.2f} RU")
            print(f"  Diferença: {abs(avg_current_ru - reference_ru):.2f} RU")
            print(f"  Erro Relativo: {abs(avg_current_ru - reference_ru)/max(reference_ru, 0.1)*100:.1f}%")

            print(f"\nEstatísticas de DP:")
            print(f"  Total de Eventos: {stats['total_events']}")
            print(f"  Corrente Máxima: {stats['max_current_ru']:.2f} RU")
            print(f"  Corrente Mínima: {stats['min_current_ru']:.2f} RU")
            print(f"  Desvio Padrão: {stats['std_current_ru']:.2f} RU")
            print(f"  Eventos Positivos: {stats['positive_count']} (Média: {stats['positive_avg_ru']:.2f} RU)")
            print(f"  Eventos Negativos: {stats['negative_count']} (Média: {stats['negative_avg_ru']:.2f} RU)")
            print(f"  Razão de Assimetria (Neg/Pos): {stats['asymmetry_ratio']:.2f}")

        self.results = comparison_results
        return comparison_results

    def plot_comparison(self, E_peak: float, frequency: float, num_cycles: int):
        """Plota comparação visual entre defeitos (similar à Figura 2)"""

        fig = plt.figure(figsize=(18, 12))

        # Plotar cada defeito
        for idx, defect_type in enumerate(self.defect_types):
            result = self.results[defect_type]
            measurements = result['measurements']
            avg_current_ru = result['avg_current_ru']
            reference_ru = result['reference_ru']
            reference_info = result['reference_info']

            # Gráfico 1: PRPD (similar à Figura 2 esquerda/direita)
            ax = plt.subplot(3, 4, idx*4 + 1)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                # Separar semiciclos
                pos_phases = [m.phase for m in measurements if 0 <= m.phase <= 180]
                pos_currents = [m.current_ru for m in measurements if 0 <= m.phase <= 180]
                neg_phases = [m.phase for m in measurements if 180 < m.phase <= 360]
                neg_currents = [m.current_ru for m in measurements if 180 < m.phase <= 360]

                ax.scatter(pos_phases, pos_currents, color='red', alpha=0.6, s=25, label='Pos')
                ax.scatter(neg_phases, neg_currents, color='blue', alpha=0.6, s=25, label='Neg')

            # Adicionar tensão AC
            phase_range = np.linspace(0, 360, 1000)
            voltage = np.sin(np.radians(phase_range))
            ax_twin = ax.twinx()
            ax_twin.plot(phase_range, voltage, 'k--', linewidth=1, alpha=0.3)

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente (RU)', fontsize=10, color='black')
            ax_twin.set_ylabel('Tensão (p.u.)', fontsize=9, color='gray')
            ax.set_title(f'{reference_info["label"]}: {reference_info["description"]}\nPRPD (Figura 2)',
                        fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 360)
            ax.legend(loc='upper right', fontsize=8)

            # Gráfico 2: Histograma de Fase com RU
            ax = plt.subplot(3, 4, idx*4 + 2)

            if measurements:
                phases = [m.phase for m in measurements]
                currents = [m.current_ru for m in measurements]

                phase_bins = np.linspace(0, 360, 37)
                hist_ru, _ = np.histogram(phases, bins=phase_bins, weights=currents)

                ax.bar(phase_bins[:-1], hist_ru, width=10, color='green', alpha=0.7, edgecolor='black')

            ax.set_xlabel('Fase (°)', fontsize=10)
            ax.set_ylabel('Corrente Integrada (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nHistograma de Corrente', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')

            # Gráfico 3: Tabela I - Comparação de RU
            ax = plt.subplot(3, 4, idx*4 + 3)

            categories = ['Medido', 'Referência\n(Tabela I)']
            values = [avg_current_ru, reference_ru]
            colors = ['#3498db', '#e74c3c']

            bars = ax.bar(categories, values, color=colors, alpha=0.8, edgecolor='black', width=0.6)

            # Adicionar valores nas barras
            for bar, value in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.2f}\nRU',
                       ha='center', va='bottom', fontsize=11, fontweight='bold')

            ax.set_ylabel('Corrente (RU)', fontsize=10)
            ax.set_title(f'{reference_info["label"]}\nTabela I do Artigo', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(0, max(values) * 1.3)

            # Gráfico 4: Estatísticas detalhadas
            ax = plt.subplot(3, 4, idx*4 + 4)
            ax.axis('off')

            stats = result['statistics']
            stats_text = f"""
{reference_info['label']} - {reference_info['description']}

TABELA I - QUANTIFICAÇÃO EM RU:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I (RU) Medido: {avg_current_ru:.2f}
I (RU) Referência: {reference_ru:.2f}
Diferença: {abs(avg_current_ru - reference_ru):.2f}

ESTATÍSTICAS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total de Eventos: {stats['total_events']}
Corrente Máxima: {stats['max_current_ru']:.2f} RU
Corrente Mínima: {stats['min_current_ru']:.2f} RU
Desvio Padrão: {stats['std_current_ru']:.2f} RU

SEMICICLOS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Positivos: {stats['positive_count']}
  Média: {stats['positive_avg_ru']:.2f} RU
Negativos: {stats['negative_count']}
  Média: {stats['negative_avg_ru']:.2f} RU
Assimetria: {stats['asymmetry_ratio']:.2f}
            """

            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                   fontsize=8.5, verticalalignment='top', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.suptitle(
            f'Comparação de Defeitos - Quantificação em Unidades Relativas (RU)\n'
            f'Conforme Figura 2 e Tabela I do Artigo\n'
            f'E_peak: {E_peak/1e6:.1f} MV/m | Freq: {frequency} Hz | Ciclos: {num_cycles}',
            fontsize=13, fontweight='bold', y=0.995
        )

        plt.tight_layout(rect=[0, 0, 1, 0.99])
        plt.savefig('defect_comparison_ru_quantification.png', dpi=150, bbox_inches='tight')
        print("\n✓ Gráfico salvo: defect_comparison_ru_quantification.png")
        plt.show()

    def generate_table_i_report(self):
        """Gera relatório formatado como Tabela I do artigo"""

        print(f"\n{'='*80}")
        print(f"TABELA I - CORRENTE EM UNIDADES RELATIVAS (RU)")
        print(f"Medições de Descarga Parcial com Antena EM")
        print(f"{'='*80}\n")

        # Cabeçalho da tabela
        print(f"{'Slice':<20} {'C-1A':<15} {'CE-1E':<15} {'CE-1D':<15}")
        print(f"{'Descrição':<20} {'Cavidade Int.':<15} {'Desc. Superf.':<15} {'Corona':<15}")
        print("─" * 80)

        # Linha de corrente medida
        print(f"{'I (RU) - Medido':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['avg_current_ru']:<15.2f}", end='')
        print()

        # Linha de referência
        print(f"{'I (RU) - Referência':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            print(f"{result['reference_ru']:<15.2f}", end='')
        print()

        # Linha de diferença
        print(f"{'Diferença':<20}", end='')
        for defect_type in self.defect_types:
            result = self.results[defect_type]
            diff = abs(result['avg_current_ru'] - result['reference_ru'])
            print(f"{diff:<15.2f}", end='')
        print()

        print("\n" + "─" * 80)
        print("\nINTERPRETAÇÃO (Conforme Seção III do Artigo):\n")

        # Encontrar maior corrente
        max_current = max(
            (self.results[dt]['avg_current_ru'], dt)
            for dt in self.defect_types
        )

        print(f"• Maior Atividade de DP: {max_current[1]} ({max_current[0]:.2f} RU)")
        print(f"  → Maior concentração de corrente de descarga")

        # Encontrar maior assimetria
        max_asym = max(
            (self.results[dt]['statistics']['asymmetry_ratio'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Assimetria: {max_asym[1]} ({max_asym[0]:.2f})")
        print(f"  → Maior concentração de eventos em semiciclo negativo")

        # Encontrar maior número de eventos
        max_events = max(
            (self.results[dt]['statistics']['total_events'], dt)
            for dt in self.defect_types
        )

        print(f"\n• Maior Número de Eventos: {max_events[1]} ({max_events[0]} eventos)")
        print(f"  → Maior frequência de descargas parciais")


if __name__ == '__main__':
    print("╔════════════════════════════════════════════════════════════╗")
    print("║  Comparação de Defeitos - Quantificação em RU             ║")
    print("║  Conforme Tabela I do Artigo                              ║")
    print("╚════════════════════════════════════════════════════════════╝")

    # Parâmetros
    E_peak = 4.0e6  # V/m
    frequency = 60  # Hz
    num_cycles = 100

    # Executar análise
    analyzer = DefectComparisonWithRU()
    results = analyzer.run_comparison(E_peak, frequency, num_cycles)

    # Plotar comparação
    analyzer.plot_comparison(E_peak, frequency, num_cycles)

    # Gerar Tabela I
    analyzer.generate_table_i_report()

    print(f"\n{'='*80}")
    print("✓ Análise concluída com sucesso!")
    print(f"{'='*80}\n")